## Classification multi-classes à label unique

L'extension naturelle de la classification binaire est la classification multi-classes.

On aborde d'abord la classification multi-classes à label unique, qui suppose que chaque exemple est assigné à une seule classe, et une seule. Pour l'illustrer, on utilise le jeu de données Iris, une classification en trois classes mutuellement exclusives. Notons ces classes $A$, $B$ et $C$.

On pourrait entraîner trois prédicats unaires séparés, $A(x)$, $B(x)$ et $C(x)$, mais il s'avère plus efficace de modéliser ce problème avec un seul prédicat binaire $P(x,l)$, où $l$ est une variable désignant une classe parmi $A$, $B$ ou $C$. Cette syntaxe permet d'écrire des énoncés quantifiant directement sur les classes, comme $\forall x\,(\exists l\,P(x,l))$, qui affirme que chaque exemple doit se voir attribuer au moins une classe, une formulation qui n'aurait aucun sens avec trois prédicats indépendants, puisque les classes ne seraient alors pas les valeurs d'une variable qu'on peut faire varier avec un quantificateur.

Puisque les classes sont mutuellement exclusives, la dernière couche du MLP représentant $P(x,l)$ utilise une activation softmax plutôt qu'une sigmoïde, afin d'apprendre directement les probabilités de $A$, $B$ et $C$.

Pour cette tâche, LTN utilise le langage et le grounding suivants.

**Domaines :**
- $items$, désignant les exemples du jeu de données Iris ;
- $labels$, désignant les étiquettes de classe.

**Variables :**
- $x_A, x_B, x_C$ pour les exemples positifs des classes $A$, $B$ et $C$ ;
- $x$ pour l'ensemble des exemples ;
- $D(x_A) = D(x_B) = D(x_C) = D(x) = items$.

**Constantes :**
- $l_A, l_B, l_C$, les étiquettes des classes $A$ (Iris setosa), $B$ (Iris virginica), $C$ (Iris versicolor) ;
- $D(l_A) = D(l_B) = D(l_C) = labels$.

**Prédicats :**
- $P(x,l)$, exprimant que l'exemple $x$ est classé comme $l$ ;
- $D_{in}(P) = items, labels$.

**Axiomes :**
- $\forall x_A\,P(x_A,l_A)$ : tous les exemples de la classe $A$ doivent recevoir le label $l_A$ ;
- $\forall x_B\,P(x_B,l_B)$ : tous les exemples de la classe $B$ doivent recevoir le label $l_B$ ;
- $\forall x_C\,P(x_C,l_C)$ : tous les exemples de la classe $C$ doivent recevoir le label $l_C$.

On ne trouve pas ici de règle d'exclusivité du type $\forall x\,(P(x,l_A)\implies(\lnot P(x,l_B)\land\lnot P(x,l_C)))$ : une telle règle serait automatiquement vraie, quel que soit l'état d'entraînement du réseau, puisque cette contrainte est déjà imposée structurellement par le grounding de $P$ ci-dessous, plus précisément par la fonction softmax. L'écrire comme axiome logique serait donc redondant.

**Grounding :**
- $\mathcal{G}(items)=\mathbb{R}^4$ : chaque exemple est décrit par 4 caractéristiques, la longueur et la largeur des sépales et des pétales, en centimètres ;
- $\mathcal{G}(labels)=\mathbb{N}^3$ : les classes sont représentées en encodage one-hot ;
- $\mathcal{G}(x_A)\in\mathbb{R}^{m_1\times4}$, $\mathcal{G}(x_B)\in\mathbb{R}^{m_2\times4}$, $\mathcal{G}(x_C)\in\mathbb{R}^{m_3\times4}$ : des séquences d'exemples propres à chaque classe, sans que leurs tailles respectives $m_1,m_2,m_3$ aient besoin d'être égales ;
- $\mathcal{G}(x)\in\mathbb{R}^{(m_1+m_2+m_3)\times4}$ : la séquence de l'ensemble des exemples ;
- $\mathcal{G}(l_A)=[1,0,0]$, $\mathcal{G}(l_B)=[0,1,0]$, $\mathcal{G}(l_C)=[0,0,1]$ ;
- $\mathcal{G}(P\mid\theta):x,l\mapsto l^\top\cdot\text{softmax}(\text{MLP}_\theta(x))$, où le MLP possède trois neurones de sortie, un par classe, et où $\cdot$ désigne le produit scalaire, qui sert ici à sélectionner la probabilité correspondant à la classe désignée par $l$. Puisque $l$ ne contient qu'un seul $1$, à la position de la vraie classe, et des $0$ ailleurs, ce produit scalaire extrait exactement la probabilité de cette classe, en annulant toutes les autres, exactement l'opération déjà rencontrée en code sous la forme `torch.sum(prob * l, dim=1)`.

### Jeu de données

Importons maintenant le jeu de données.

Le jeu de données Iris comporte trois classes de 50 exemples chacune. Chaque exemple est décrit par 4 caractéristiques.

Le chargement se fait avec `pandas` : on lit les deux fichiers CSV, déjà séparés en entraînement et test, puis on extrait la colonne des étiquettes (`"species"`) du reste des données via `.pop()`. Les caractéristiques sont converties en tenseurs flottants, les étiquettes en tenseurs d'entiers, ces derniers représentant à ce stade les classes sous forme d'indices (0, 1 ou 2), avant leur conversion en encodage one-hot un peu plus loin dans le notebook.

In [1]:
import torch
import pandas as pd

train_data = pd.read_csv("datasets/iris_training.csv")
test_data = pd.read_csv("datasets/iris_test.csv")

train_labels = train_data.pop("species")
test_labels = test_data.pop("species")

train_data = torch.tensor(train_data.to_numpy()).float()
test_data = torch.tensor(test_data.to_numpy()).float()
train_labels = torch.tensor(train_labels.to_numpy()).long()
test_labels = torch.tensor(test_labels.to_numpy()).long()

### Paramétrage LTN

Pour définir notre base de connaissances (les axiomes), il faut définir le prédicat $P$, les constantes $l_A$, $l_B$, $l_C$, le quantificateur universel, et l'opérateur `SatAgg`.

Pour le quantificateur, on utilise la configuration produit stable, déjà présentée dans les tutoriels.

Pour le prédicat $P$, on construit deux modèles distincts. Le premier implémente un MLP qui renvoie les logits pour les trois classes du jeu de données Iris, à partir d'un exemple $x$ donné en entrée. Le second prend en entrée un exemple étiqueté $(x,l)$, calcule les logits à l'aide du premier modèle, puis renvoie la prédiction (via softmax) pour la classe $l$.

Ces deux modèles séparés sont nécessaires parce qu'on a besoin à la fois des logits et des probabilités, mais pour deux usages différents : les logits serviront à calculer la précision de classification, tandis que les probabilités, elles, seront interprétées comme des degrés de vérité pour calculer le niveau de satisfaction de la base de connaissances.

Les constantes $l_A$, $l_B$ et $l_C$ représentent les labels one-hot des trois classes, exactement comme on l'a déjà vu dans la définition du grounding de cette tâche.

`SatAgg` est défini à l'aide de l'agrégateur `pMeanError`.

In [2]:
import ltn

# on définit les constantes
l_A = ltn.Constant(torch.tensor([1, 0, 0]))
l_B = ltn.Constant(torch.tensor([0, 1, 0]))
l_C = ltn.Constant(torch.tensor([0, 0, 1]))

# on définit le prédicat P
class MLP(torch.nn.Module):
    """
    Ce modèle renvoie les logits des classes pour un exemple donné en entrée. Il ne calcule pas le softmax,
    la sortie n'est donc pas normalisée.
    Ce choix permet de séparer le calcul de la précision de celui du niveau de satisfaction.
    Parcourir l'exemple permet de bien comprendre pourquoi.
    """
    def __init__(self, layer_sizes=(4, 16, 16, 8, 3)):
        super(MLP, self).__init__()
        self.elu = torch.nn.ELU()
        self.dropout = torch.nn.Dropout(0.2)
        self.linear_layers = torch.nn.ModuleList([torch.nn.Linear(layer_sizes[i - 1], layer_sizes[i])
                                                  for i in range(1, len(layer_sizes))])

    def forward(self, x, training=False):
        """
        Phase forward du réseau pour cette tâche de classification multi-classes.
        Renvoie les logits des classes pour l'exemple x.

        :param x: les caractéristiques de l'exemple
        :param training: indique si le réseau est en mode entraînement (dropout appliqué)
                          ou en mode évaluation (dropout non appliqué)
        :return: logits pour l'exemple x
        """
        for layer in self.linear_layers[:-1]:
            x = self.elu(layer(x))
            if training:
                x = self.dropout(x)
        logits = self.linear_layers[-1](x)
        return logits


class LogitsToPredicate(torch.nn.Module):
    """
    Ce modèle encapsule un modèle de logits, c'est-à-dire un modèle qui calcule les logits des classes
    à partir d'un exemple x donné en entrée. L'idée est de garder logits et probabilités séparés :
    le modèle de logits renvoie les logits pour un exemple, tandis que ce modèle-ci renvoie les
    probabilités à partir de ces logits.

    Concrètement, il prend en entrée un exemple x et un label de classe l. Il applique le modèle de logits
    à x pour obtenir les logits, puis une fonction softmax pour obtenir les probabilités par classe.
    Enfin, il ne renvoie que la probabilité associée au label l donné.
    """
    def __init__(self, logits_model):
        super(LogitsToPredicate, self).__init__()
        self.logits_model = logits_model
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, x, l, training=False):
        logits = self.logits_model(x, training=training)
        probs = self.softmax(logits)
        out = torch.sum(probs * l, dim=1)
        return out

mlp = MLP()
P = ltn.Predicate(LogitsToPredicate(mlp))

# on définit les connecteurs, quantificateurs, et SatAgg
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
SatAgg = ltn.fuzzy_ops.SatAgg()

### Utilitaires

Définissons maintenant quelques classes et fonctions utilitaires.

On définit un chargeur de données PyTorch standard, qui prend en entrée le dataset et renvoie un générateur de batchs. On en a besoin de deux instances : une pour les données d'entraînement, une pour les données de test.

On définit ensuite des fonctions pour évaluer les performances du modèle. Le modèle est évalué sur l'ensemble de test à l'aide des métriques suivantes :
* le niveau de satisfaction de la base de connaissances, qui mesure la capacité de LTN à satisfaire les règles logiques ;
* la précision de classification, qui mesure directement la qualité des prédictions.

In [3]:
from sklearn.metrics import accuracy_score
import numpy as np

# chargeur de données PyTorch standard, pour l'entraînement et le test du modèle
class DataLoader(object):
    def __init__(self,
                 data,
                 labels,
                 batch_size=1,
                 shuffle=True):
        self.data = data
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.data.shape[0] / self.batch_size))

    def __iter__(self):
        n = self.data.shape[0]
        idxlist = list(range(n))
        if self.shuffle:
            np.random.shuffle(idxlist)

        for _, start_idx in enumerate(range(0, n, self.batch_size)):
            end_idx = min(start_idx + self.batch_size, n)
            data = self.data[idxlist[start_idx:end_idx]]
            labels = self.labels[idxlist[start_idx:end_idx]]

            yield data, labels


# définition des métriques d'évaluation du modèle

# calcule le niveau global de satisfaction de la base de connaissances, pour le chargeur donné (train ou test)
def compute_sat_level(loader):
    mean_sat = 0
    for data, labels in loader:
        x_A = ltn.Variable("x_A", data[labels == 0])
        x_B = ltn.Variable("x_B", data[labels == 1])
        x_C = ltn.Variable("x_C", data[labels == 2])
        mean_sat += SatAgg(
            Forall(x_A, P(x_A, l_A)),
            Forall(x_B, P(x_B, l_B)),
            Forall(x_C, P(x_C, l_C))
        )
    mean_sat /= len(loader)
    return mean_sat

# calcule la précision globale des prédictions du modèle entraîné, pour le chargeur donné (train ou test)
def compute_accuracy(loader):
    mean_accuracy = 0.0
    for data, labels in loader:
        predictions = mlp(data).detach().numpy()
        predictions = np.argmax(predictions, axis=1)
        mean_accuracy += accuracy_score(labels, predictions)

    return mean_accuracy / len(loader)

# création des chargeurs d'entraînement et de test
train_loader = DataLoader(train_data, train_labels, 64, shuffle=True)
test_loader = DataLoader(test_data, test_labels, 64, shuffle=False)

### Apprentissage

Notons $D$ l'ensemble complet des exemples du dataset. L'objectif, pour la base de connaissances $\mathcal{K}=\{\forall x_A\,P(x_A,l_A),\ \forall x_B\,P(x_B,l_B),\ \forall x_C\,P(x_C,l_C)\}$, est donné par $\text{SatAgg}_{\phi\in\mathcal{K}}\ \mathcal{G}_{\theta,\,x\leftarrow D}(\phi)$.

Cette notation se lit exactement comme dans l'exemple précédent : le grounding $\mathcal{G}$, indicé par $\theta$ (les poids actuels du réseau $P$) et par $x\leftarrow D$ (les variables sont remplies avec les données de $D$), donne le degré de vérité de $\phi$ à cet instant de l'entraînement.

En pratique, l'optimiseur utilise la fonction de perte suivante :

$$L = 1-\text{SatAgg}_{\phi\in\mathcal{K}}\ \mathcal{G}_{\theta,\,x\leftarrow B}(\phi)$$

où $B$ est un mini-batch tiré de $D$, exactement le même principe déjà détaillé pour la classification binaire.

Dans ce qui suit, on entraîne notre LTN sur la tâche de classification multi-classes à label unique, en utilisant la satisfaction de la base de connaissances comme objectif. Autrement dit, on cherche à apprendre les paramètres $\theta$ du prédicat binaire $P$ de façon à satisfaire au mieux les trois axiomes de la base de connaissances. Le modèle est entraîné sur 500 epochs, avec l'optimiseur `Adam`.

La figure suivante montre le graphe de calcul LTN associé à cette tâche.

![Graphe de calcul](./images/multi-class-single-label-classification.png)

In [4]:
optimizer = torch.optim.Adam(P.parameters(), lr=0.001)

for epoch in range(500):
    train_loss = 0.0
    for batch_idx, (data, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        # on groundes les variables avec les données du batch courant
        x_A = ltn.Variable("x_A", data[labels == 0])  # exemples de classe A
        x_B = ltn.Variable("x_B", data[labels == 1])  # exemples de classe B
        x_C = ltn.Variable("x_C", data[labels == 2])  # exemples de classe C
        sat_agg = SatAgg(
            Forall(x_A, P(x_A, l_A, training=True)),
            Forall(x_B, P(x_B, l_B, training=True)),
            Forall(x_C, P(x_C, l_C, training=True))
        )
        loss = 1. - sat_agg
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss = train_loss / len(train_loader)

    # on affiche les métriques toutes les 20 epochs
    if epoch % 20 == 0:
        print(" epoch %d | loss %.4f | Train Sat %.3f | Test Sat %.3f | Train Acc %.3f | Test Acc %.3f"
              %(epoch, train_loss, compute_sat_level(train_loader), compute_sat_level(test_loader),
                    compute_accuracy(train_loader), compute_accuracy(test_loader)))

C:\temp\ipykernel_50556\3001372085.py:24: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:823.)
  print(" epoch %d | loss %.4f | Train Sat %.3f | Test Sat %.3f | Train Acc %.3f | Test Acc %.3f"


 epoch 0 | loss 0.6717 | Train Sat 0.334 | Test Sat 0.334 | Train Acc 0.352 | Test Acc 0.267
 epoch 20 | loss 0.6232 | Train Sat 0.386 | Test Sat 0.387 | Train Acc 0.353 | Test Acc 0.267
 epoch 40 | loss 0.4987 | Train Sat 0.550 | Test Sat 0.553 | Train Acc 0.839 | Test Acc 0.767
 epoch 60 | loss 0.4033 | Train Sat 0.693 | Test Sat 0.700 | Train Acc 0.967 | Test Acc 0.967
 epoch 80 | loss 0.3110 | Train Sat 0.771 | Test Sat 0.782 | Train Acc 0.982 | Test Acc 0.967
 epoch 100 | loss 0.2506 | Train Sat 0.812 | Test Sat 0.826 | Train Acc 0.983 | Test Acc 0.967
 epoch 120 | loss 0.1968 | Train Sat 0.838 | Test Sat 0.845 | Train Acc 0.983 | Test Acc 0.967
 epoch 140 | loss 0.2185 | Train Sat 0.853 | Test Sat 0.855 | Train Acc 0.983 | Test Acc 0.967
 epoch 160 | loss 0.2005 | Train Sat 0.857 | Test Sat 0.860 | Train Acc 0.983 | Test Acc 0.967
 epoch 180 | loss 0.1745 | Train Sat 0.873 | Test Sat 0.860 | Train Acc 0.991 | Test Acc 0.967
 epoch 200 | loss 0.1860 | Train Sat 0.878 | Test Sat 0.

Les variables $x_A$, $x_B$ et $x_C$ sont groundées batch par batch, avec de nouvelles données provenant du chargeur à chaque itération, exactement ce que représente la notation $\mathcal{G}_{x\leftarrow B}(\phi(x))$, où $B$ est le mini-batch fourni par le chargeur.

`SatAgg` prend ici en entrée les trois axiomes et renvoie une seule valeur de vérité, interprétée comme le niveau de satisfaction global de la base de connaissances.

On observe qu'après seulement 80 epochs, la précision sur les données de test avoisine 1, ce qui montre la capacité de LTN à apprendre cette tâche de classification multi-classes en n'utilisant que la satisfaction d'une base de connaissances comme objectif.